In [109]:
import os
import glob
import pandas as pd
import numpy as np
from scipy.stats import linregress
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.base import clone
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna


In [3]:
data_dir = "./pump_fun_dataset"  
divers_path = os.path.join(data_dir, "token_info_onchain_divers.csv")
dune_path = os.path.join(data_dir, "dune_token_info.csv")
train_path = os.path.join(data_dir, "train.csv")
train = pd.read_csv(train_path)

dune_unique = pd.read_csv(dune_path)
divers_unique = pd.read_csv(divers_path)
#dune_unique = dune_extended.drop_duplicates(subset=['token_mint_address'])
#divers_unique = divers_extended.drop_duplicates(subset=['mint'])


C:\Users\kevin\AppData\Local\Temp\ipykernel_23384\1456146577.py:8: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  divers_unique = pd.read_csv(divers_path)


In [4]:
def process(train):
    # Load and concatenate all behavioral chunk files
    chunk_files = glob.glob(os.path.join(data_dir, "chunk*.csv"))
    chunks = [pd.read_csv(f) for f in chunk_files]
    chunk_all = pd.concat(chunks, ignore_index=True)

    chunk_all['block_time'] = pd.to_datetime(chunk_all['block_time'],utc=True).dt.tz_localize(None)
    dune_unique['created_at'] = pd.to_datetime(dune_unique['created_at'],utc=True).dt.tz_localize(None)

    # 3. Compute slot offset
    chunk_all = chunk_all.merge(train[['mint','slot_min']], left_on='base_coin', right_on='mint', how='left')
    chunk_all['slot_offset'] = chunk_all['slot'] - chunk_all['slot_min']

    # 4. Basic aggregates
    agg_funcs = {
        'tx_idx': ['count'], # Total transactions
        'block_time': ['min', 'max'], # First and last transaction time
        'slot': ['min', 'max', 'nunique'], # First, last, and number of unique blocks with activity
        'signing_wallet': ['nunique'], # Number of unique traders
        'quote_coin_amount': ['sum', 'mean', 'std', 'max'], # SOL volume stats
        'base_coin_amount': ['sum', 'mean', 'std', 'max'], # Token volume stats
        'virtual_sol_balance_after': ['last', 'max', 'min', 'mean', 'std'], # SOL balance proxy
        'virtual_token_balance_after': ['last', 'max', 'min', 'mean', 'std'] # Token balance proxy
    }
    chunk_feat = chunk_all.groupby('base_coin').agg(agg_funcs)
    chunk_feat.columns = ['_'.join(col) for col in chunk_feat.columns]
    chunk_feat = chunk_feat.reset_index().rename(columns={'base_coin': 'mint'})

    # 5. Slot-windowed SOL volumes
    def window_sum(low, high):
        w = chunk_all[(chunk_all['slot_offset'] >= low) & (chunk_all['slot_offset'] < high)]
        return w.groupby('base_coin')['quote_coin_amount'].sum().rename(f'sol_sum_{low}_{high}')
    w0_10 = window_sum(0, 10)
    w10_30 = window_sum(10, 30)
    w30_100 = window_sum(30, 100)

    # 6. Time-windowed SOL volume (first 60 seconds)
    mints = dune_unique[['token_mint_address','created_at']].rename(columns={'token_mint_address':'mint'})
    chunk_all = chunk_all.merge(mints, on='mint', how='left')
    chunk_all['seconds_since_mint'] = (chunk_all['block_time'] - chunk_all['created_at']).dt.total_seconds()
    first_min = chunk_all[chunk_all['seconds_since_mint'] <= 60]
    sol_sum_60s = first_min.groupby('base_coin')['quote_coin_amount'].sum().rename('sol_sum_0_60')

    # 7. Slope of virtual SOL balance over slots
    def slope(group):
        if group['slot_offset'].nunique() < 2:
            return 0.0
        return linregress(group['slot_offset'], group['virtual_sol_balance_after']).slope
    slope_sol = chunk_all.groupby('base_coin').apply(slope).rename('sol_balance_slope')

    # 8. Buy/Sell pivot via pivot_table
    bs = chunk_all.pivot_table(
        index='base_coin', 
        columns='direction', 
        values='quote_coin_amount', 
        aggfunc=['sum','count'], 
        fill_value=0
    )
    bs.columns = [f"{stat}_{dir}" for stat,dir in bs.columns]
    bs = bs.reset_index().rename(columns={'base_coin':'mint'})

    # 9. Time-to-first-trade
    first_trade = chunk_all.groupby('base_coin')['slot'].min().rename('first_trade_slot').reset_index()
    first_trade.rename(columns = {'base_coin':'mint'}, inplace=True)
    first_trade = first_trade.merge(train[['mint','slot_min']], on='mint', how='left')
    first_trade['time_to_first_trade'] = (first_trade['first_trade_slot'] - first_trade['slot_min'] + 1).clip(lower=0)
    ttf = first_trade[['mint','time_to_first_trade']]
    
    # 1. Wallet retention: unique wallets in early vs. later windows
    w0_10_wallets = chunk_all[chunk_all['slot_offset'] < 10] \
                        .groupby('base_coin')['signing_wallet'].nunique() \
                        .rename('wallets_0_10')
    w10_30_wallets = chunk_all[(chunk_all['slot_offset'] >= 10) & (chunk_all['slot_offset'] < 30)] \
                        .groupby('base_coin')['signing_wallet'].nunique() \
                        .rename('wallets_10_30')
    wallet_retention = pd.concat([w0_10_wallets, w10_30_wallets], axis=1).fillna(0)
    wallet_retention['wallet_retention_ratio'] = (
        wallet_retention['wallets_10_30'] / 
        (wallet_retention['wallets_0_10'] + 1e-6)
    )

    # 2. Volume-to-wallet ratios for each window
    vol_wallet = chunk_all.groupby('base_coin').agg({
        'quote_coin_amount': 'sum',
        'signing_wallet': 'nunique'
    }).rename(columns={
        'quote_coin_amount': 'total_volume',
        'signing_wallet': 'total_wallets'
    })
    vol_wallet['vol_per_wallet'] = (
        vol_wallet['total_volume'] / 
        (vol_wallet['total_wallets'] + 1e-6)
    )

    # 3. Price change: (last_price - first_price) / first_price
    # first and last defined by slot_offset
    first_price = chunk_all.sort_values('slot_offset') \
                    .groupby('base_coin').first()
    last_price  = chunk_all.sort_values('slot_offset') \
                    .groupby('base_coin').last()
    # assume price = quote_coin_amount / base_coin_amount
    price_df = pd.DataFrame({
        'first_price': first_price['quote_coin_amount'] / first_price['base_coin_amount'],
        'last_price':  last_price['quote_coin_amount'] / last_price['base_coin_amount']
    })
    price_df['price_change_pct'] = (
        (price_df['last_price'] - price_df['first_price']) 
        / (price_df['first_price'] + 1e-6)
    )
    price_df['price_change_pct'] = price_df['price_change_pct'].replace(
    [np.inf, -np.inf], np.nan
                            ).fillna(0)

    # 4. First-trade direction flag (1=buy, 0=sell)
    first_trade_dir = chunk_all.sort_values('slot_offset') \
                         .groupby('base_coin')['direction'] \
                         .first() \
                         .map({'buy': 1, 'sell': 0}) \
                         .rename('first_trade_is_buy')

    # 5. Pump-dump spike: ratio of max volume in a slot to mean volume
    slot_vol = chunk_all.groupby(['base_coin','slot'])['quote_coin_amount'].sum()
    slot_stats = slot_vol.groupby('base_coin').agg(['max','mean'])
    slot_stats['vol_spike_ratio'] = slot_stats['max'] / (slot_stats['mean'] + 1e-6)
    vol_spike = slot_stats['vol_spike_ratio'].rename('vol_spike_ratio')

    # Merge all of these together on 'base_coin'
    feats = pd.concat([
        wallet_retention,
        vol_wallet['vol_per_wallet'],
        price_df['price_change_pct'],
        first_trade_dir,
        vol_spike
    ], axis=1).reset_index().rename(columns={'base_coin':'mint'})

    # 10. Unique wallet count
    wallet_counts = chunk_all.groupby('base_coin')['signing_wallet'].nunique().rename('unique_wallets').reset_index()
    wallet_counts = wallet_counts.rename(columns={'base_coin':'mint'})

    features = [w0_10, w10_30, w30_100, sol_sum_60s, slope_sol]
    feature_df = pd.concat(features, axis=1).reset_index().rename(columns={'base_coin':'mint'})
    chunk_features = chunk_feat.merge(feature_df, on='mint', how='left') \
                                .merge(bs, on='mint', how='left') \
                                .merge(ttf, on='mint', how='left') \
                                .merge(wallet_counts, on='mint', how='left') \
                                .merge(feats, on='mint', how='left') 

    # 12. Merge into train_processed
    train_processed = train.merge(chunk_features, on='mint', how='left')
    train_processed = train_processed.merge(
        dune_unique.rename(columns={'token_mint_address':'mint'}), on='mint', how='left'
    )
    train_processed = train_processed.merge(divers_unique, on='mint', how='left')
    train_processed[['sol_sum_0_10','sol_sum_10_30','sol_sum_30_100']] = \
        train_processed[['sol_sum_0_10','sol_sum_10_30','sol_sum_30_100']].fillna(0)
    
    return train_processed

train_processed = process(train)



    

C:\Users\kevin\AppData\Local\Temp\ipykernel_23384\3540373249.py:49: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  slope_sol = chunk_all.groupby('base_coin').apply(slope).rename('sol_balance_slope')


In [5]:
train_processed = train_processed.drop(columns=[
    'mint', 'slot_min', 'slot_graduated','is_valid',
    'token_uri', 'init_tx', 'url', 'name', 'symbol_x','Unnamed: 0','symbol_y','name_y','name_x','tx_idx','base_coin','direction','block_time',
    'version','pf_program_index','curve_address','bundle_structure','amount_of_lookup_writes','amount_of_lookup_reads','amount_of_instructions'
],errors='ignore')
# Count of missing values per column
na_counts = train_processed.isna().sum()
# Percentage of missing values per column
na_pct    = (train_processed.isna().mean() * 100).round(2)
# Combine into one table
missing = pd.concat([na_counts, na_pct], axis=1)
missing.columns = ['missing_count', 'missing_pct']
missing = missing.sort_values('missing_count', ascending=False)

print(missing)

                                  missing_count  missing_pct
direct_pf_invocation                     168469        26.34
creation_ix_index                        168469        26.34
dev_balance                              168469        26.34
bundled_buys_count                       168469        26.34
bundled_buys                             168469        26.34
gas_used                                 168469        26.34
bundle_size                              168469        26.34
creator                                  168469        26.34
slot                                     168469        26.34
sol_sum_0_60                             101546        15.88
created_at                               101546        15.88
decimals                                 101546        15.88
virtual_token_balance_after_std           19143         2.99
virtual_sol_balance_after_std             19143         2.99
base_coin_amount_std                      19143         2.99
quote_coin_amount_std   

In [6]:
X = train_processed.drop(columns=['has_graduated'])
y = train_processed['has_graduated'].astype(int)
cols_to_drop = train_processed.select_dtypes(include=['object', 'datetime64[ns]']).columns.tolist()
X = X.drop(columns=cols_to_drop)
X = X.fillna(0)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,    
    stratify=y,
    random_state=42
)

In [7]:
test = pd.read_csv(os.path.join(data_dir, "test_unlabeled.csv"))
test_processed = process(test)
test_processed = test_processed.drop_duplicates(
    subset='mint',
    keep='first'    
)
drop_cols = [
    'mint', 'slot_min', 'slot_graduated','is_valid',
    'token_uri', 'init_tx', 'url', 'name', 'symbol_x','Unnamed: 0','symbol_y','name_y','name_x','tx_idx','base_coin','direction','block_time',
    'version','pf_program_index','curve_address','bundle_structure','amount_of_lookup_writes','amount_of_lookup_reads','amount_of_instructions'
]
X_test = test_processed.drop(columns=drop_cols, errors='ignore')
X_test = X_test.fillna(0)
cols_to_drop = train_processed.select_dtypes(include=['object', 'datetime64[ns]']).columns.tolist()
X_test = X_test.drop(columns=cols_to_drop)






C:\Users\kevin\AppData\Local\Temp\ipykernel_23384\3540373249.py:49: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  slope_sol = chunk_all.groupby('base_coin').apply(slope).rename('sol_balance_slope')
C:\Users\kevin\AppData\Local\Temp\ipykernel_23384\503097773.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test = X_test.fillna(0)


In [ ]:
best_params = {}
def objective(trial):
    params = {
        "n_estimators":    trial.suggest_int("n_estimators", 100, 1000),
        "max_depth":       trial.suggest_int("max_depth", 3, 12),
        "learning_rate":   trial.suggest_float("lr", 1e-3, 0.3, log=True),
        "subsample":       trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":trial.suggest_float("colsample", 0.5, 1.0),
        "gamma":           trial.suggest_float("gamma", 0.0, 5.0),
        "min_child_weight":trial.suggest_int("mcw", 1, 10),
        "reg_alpha":       trial.suggest_float("alpha", 1e-8, 10.0, log=True),
        "reg_lambda":      trial.suggest_float("lambda",1e-8, 10.0, log=True),
    }
    clf = XGBClassifier(
        tree_method="hist",
        device="cuda",
        random_state=42,
        eval_metric="logloss",
        **params
    )
    # train on full training split
    clf.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    # evaluate on the held-out val set
    proba = clf.predict_proba(X_val)[:,1]
    return log_loss(y_val, proba)  

# 3) Run the study
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)
print("Val log-loss:", study.best_value)
best_params['XGB'] = study.best_params.copy()
#0.03910539796468443

In [ ]:
def objective(trial):
    params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 2000),
            "num_leaves":   trial.suggest_int("leaves", 20, 256),
            "max_depth":    trial.suggest_int("max_depth", 3, 16),
            "learning_rate":trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "subsample":    trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            "reg_alpha":    trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda":   trial.suggest_float("reg_lambda",1e-8, 10.0, log=True),
    }
    clf = LGBMClassifier(
            device="gpu",
            gpu_platform_id=0,
            gpu_device_id=0,
            random_state=42,
            eval_metric="logloss",
            **params
        )
    # train on full training split
    clf.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
    )
    # evaluate on the held-out val set
    proba = clf.predict_proba(X_val)[:,1]
    return log_loss(y_val, proba)  

# 3) Run the study
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)
print("Val log-loss:", study.best_value)
best_params['LGB'] = study.best_params.copy()
#0.03906

In [ ]:
def objective(trial):
    params = {
            "iterations":         trial.suggest_int("iterations", 100, 2000),
            "depth":              trial.suggest_int("depth", 4, 10),
            "learning_rate":      trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "l2_leaf_reg":        trial.suggest_float("l2_leaf_reg", 1e-8, 100.0, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "border_count":       trial.suggest_int("border_count", 32, 255),
        }
    clf = CatBoostClassifier(
            task_type="GPU",
            devices="0",
            verbose=0,
            allow_writing_files=False,
            random_state=42,
            early_stopping_rounds=100,
            eval_metric='Logloss',
            **params
        )
    # train on full training split
    clf.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    # evaluate on the held-out val set
    proba = clf.predict_proba(X_val)[:,1]
    return log_loss(y_val, proba)  

# 3) Run the study
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)
print("Val log-loss:", study.best_value)
best_params['CB'] = study.best_params.copy()

In [ ]:
best_params = {
    "XGB": {
        "n_estimators": 641,
        "max_depth": 9,
        "learning_rate": 0.008014030248002684,    # was “lr”
        "subsample": 0.7740874400748888,
        "colsample_bytree": 0.6043062275068197,    # was “colsample”
        "gamma": 1.3200421839328649,
        "min_child_weight": 5,                     # was “mcw”
        "reg_alpha": 3.1609630747035705e-07,       # was “alpha”
        "reg_lambda": 1.1544757677854744e-05       # was “lambda”
    },
    "LGB": {
        "n_estimators": 493,                       # was “lgb_n_estimators”
        "num_leaves": 53,                          # was “lgb_leaves”
        "max_depth": 10,                           # was “lgb_max_depth”
        "learning_rate": 0.015695698881377878,     # was “lgb_lr”
        "subsample": 0.695139874793308,            # was “lgb_subsample”
        "colsample_bytree": 0.5913458062147399,    # was “lgb_colsample”
        "min_child_samples": 33,                   # was “lgb_mcs”
        "reg_alpha": 0.39971655716559434,          # was “lgb_alpha”
        "reg_lambda": 0.018097562863437434         # was “lgb_lambda”
    },
    "CB": {
        "iterations": 1550,                        # was “cb_iter”
        "depth": 9,                                # was “cb_depth”
        "learning_rate": 0.013121455484647434,     # was “cb_lr”
        "l2_leaf_reg": 5.8117197339350195,         # was “cb_l2”
        "bagging_temperature": 0.9932364767906625, # was “cb_bt”
        "border_count": 136                        # was “cb_bc”
    },

}

In [ ]:


models = {
    "XGB": XGBClassifier(
        tree_method="hist",
        device="cuda",
        random_state=42,
        eval_metric="logloss",
        **best_params['XGB']
    ), 
    "LGB": LGBMClassifier(
            device="gpu",
            gpu_platform_id=0,
            gpu_device_id=0,
            random_state=42,
            eval_metric="logloss",
            **best_params['LGB']
        ), 
    "CB": CatBoostClassifier(
            task_type="GPU",
            devices="0",
            verbose=0,
            allow_writing_files=False,
            random_state=42,
            early_stopping_rounds=100,
            eval_metric='Logloss',
            **best_params['CB']
        ),
}

for name, mdl in models.items():
    print(f"Fitting {name} on full data …")
    mdl.fit(X_train, y_train)
    print(f"Predicting {name} test probabilities …") 
    print(log_loss(y_val, mdl.predict_proba(X_val)[:, 1]))

In [ ]:
Best_K = {'XGB': 49, 'LGB': 39, 'CB': 40}
selected_models = ["XGB", "LGB","CB"] 
preds = []
for name in selected_models:
    mdl = models[name]
    K   = best_K[name]
    selector = SelectFromModel(
        estimator=mdl,
        prefit=True,
        max_features=K,
        threshold=-np.inf
    )
    # 2) reduce features
    X_tr_sel = selector.transform(X_train)
    X_va_sel = selector.transform(X_val)
    # 3) re-fit & predict
    m = clone(mdl)
    m.fit(X_tr_sel, y_train)
    proba = m.predict_proba(X_va_sel)[:, 1]
    preds.append(proba)
    print(f"{name}  →  val log-loss: {log_loss(y_val, proba):.5f}")
# 4) Ensemble by simple average over your selected models
final_pred = np.mean(preds, axis=0)
# 5) Evaluate ensemble
print("\nEnsemble (avg over {})".format(selected_models))
print("  log-loss:", log_loss(y_val, final_pred))

In [ ]:
from sklearn.ensemble import VotingClassifier

voter = VotingClassifier(
    estimators=[(name, models[name]) for name in selected_models],
    voting="soft",    

)
voter.fit(X_train, y_train)

# Evaluate
proba = voter.predict_proba(X_val)[:, 1]
print("Voting log-loss:", log_loss(y_val, proba))

In [ ]:
proba = voter.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({
    'mint':       test_processed['mint'],           
    "prediction": proba
})


submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv with", len(submission), "rows.")

[LightGBM] [Warning] Unknown parameter: eval_metric
Wrote submission.csv with 478832 rows.
